# BIODCASE Bioacoustic Classification Pipeline

This notebook documents the complete project pipeline objectively, from raw annotations to model evaluation. It follows the code already present in the repository and is designed to be runnable step by step.

Expensive operations are guarded by `RUN_*` flags. By default, the notebook inspects existing artifacts instead of rebuilding the full manifest, cache, or training run.

## 1. Pipeline Overview

The project has these main stages:

1. Read BIODCASE annotation CSV files and WAV files.
2. Build a clean event manifest with clipped event times and quality filtering.
3. Convert each event into a 3-channel spectrogram tensor.
4. Cache spectrogram tensors on disk to avoid repeated audio preprocessing.
5. Train a ResNet18 classifier using class weighting, optional focal loss, and optional weighted sampling.
6. Evaluate the best checkpoint with global, per-class, per-dataset, confusion, confidence, PR-curve, and baseline reports.
7. Inspect common errors and run inference on individual images or audio events.

In [ ]:

from pathlib import Path
import json
import sys

import pandas as pd
import yaml

from src import pipeline
from src.utils.config import load_config

try:
    display
except NameError:
    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd()
print(PROJECT_ROOT)
print("python", sys.executable)

# Explicit notebook control panel. Defaults are safe for inspection-only use.
CONFIG_PATH = Path("configs/nitro4060_bpd.yaml")
SMOKE_CONFIG_PATH = Path("configs/smoke.yaml")
MANIFEST_PATH = Path("data_manifest.csv")
DATA_ROOT = Path("biodcase_development_set")
CACHE_ROOT = Path("processed_cache")
RUNS_ROOT = Path("outputs/runs")

RUN_BUILD_MANIFEST = False
RUN_CACHE_SUMMARY = True
RUN_SMOKE_TRAINING = False
RUN_FULL_TRAINING = False
RUN_EVALUATION = False
RUN_PREDICTION = False
RUN_IMBALANCE_AUDIT = False
RUN_ERROR_EXPORT = False
RUN_TESTS = False

# Optional overrides used by later orchestration cells.
CHECKPOINT_PATH = None
EVALUATION_OUTPUT_DIR = None
ERROR_REPORT_PATH = None
ERROR_EXPORT_DIR = Path("outputs/error_samples/notebook")


## 2. Repository Layout

The important source folders are:

- `src/data`: manifest creation, spectrogram generation, cache tools, dataset class.
- `src/models`: model factory.
- `src/training`: shared loader logic, losses, train/evaluate/predict scripts.
- `src/utils`: config loading, metrics, seed and audio datetime helpers.
- `src/analysis`: post-training audits and error export helpers.
- `configs`: YAML experiment configurations.
- `tests`: unit tests for utility behavior and reporting.
- `outputs/runs`: training and evaluation artifacts.
- `processed_cache`: cached spectrogram tensors.

In [ ]:
for folder in ["src", "configs", "tests", "outputs", "processed_cache", "biodcase_development_set"]:
    path = PROJECT_ROOT / folder
    if path.exists():
        files = sum(1 for item in path.rglob("*") if item.is_file())
        print(f"{folder:28s} files={files}")
    else:
        print(f"{folder:28s} missing")


## 3. Environment And Dependencies

The base dependencies are declared in `requirements.txt`. The CUDA PyTorch stack is declared separately in `requirements-cu124.txt`, and test tooling is in `requirements-dev.txt`.

The project can run on CPU, but the training configs target CUDA by default.

In [ ]:
try:
    import torch
    import torchaudio
    import torchvision
    print("torch", torch.__version__)
    print("torchaudio", torchaudio.__version__)
    print("torchvision", torchvision.__version__)
    print("cuda_available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except Exception as exc:
    print(type(exc).__name__, exc)


## 4. Configuration Loading

YAML files can inherit from another YAML file through an `extends` key. The loader recursively merges parent and child configs, so experiment files only need to override changed values.

In [ ]:

config = load_config(CONFIG_PATH)

print("config:", CONFIG_PATH)
print("classes:", config["classes"])
print("model:", config["model"])
print("training:")
print(yaml.safe_dump(config["training"], sort_keys=False))


## 5. Manifest Creation

`src.data.build_manifest` reads all annotation CSV files for the requested splits. For each annotation row, it:

- verifies required columns;
- resolves the corresponding WAV path;
- removes duplicate events;
- reads real WAV duration;
- converts annotation datetimes to seconds relative to the WAV start time;
- clips event boundaries to available audio;
- discards events that start after audio end or contain less than the configured valid duration;
- writes a clean manifest and quality reports.

The default clean manifest is `data_manifest.csv`.

In [ ]:

if RUN_BUILD_MANIFEST:
    manifest, issues = pipeline.build_manifest(
        data_root=DATA_ROOT,
        out=MANIFEST_PATH,
        quality_report="outputs/data_quality_report.csv",
        quality_summary="outputs/data_quality_summary.csv",
        min_valid_seconds=0.5,
        splits=("train", "validation"),
    )
else:
    print("Skipped manifest rebuild. Using existing data_manifest.csv if present.")


In [ ]:

manifest = pd.read_csv(MANIFEST_PATH)

print("rows", len(manifest))
print("columns", list(manifest.columns))
display(manifest.head())


## 6. Split, Class, And Dataset Distributions

The project uses the dataset-provided `train` and `validation` directories. It does not create a random stratified split. This matters because validation contains different datasets from training, so validation measures domain transfer as well as classification accuracy.

In [ ]:
print("Rows per split")
display(manifest["split"].value_counts().sort_index().rename("count"))

print("Labels by split")
display(pd.crosstab(manifest["label"], manifest["split"]))

print("Datasets by split")
display(pd.crosstab(manifest["dataset"], manifest["split"]))

print("Quality status")
display(manifest["quality_status"].value_counts(dropna=False).rename("count"))


In [ ]:
quality_summary_path = PROJECT_ROOT / "outputs" / "data_quality_summary.csv"
quality_report_path = PROJECT_ROOT / "outputs" / "data_quality_report.csv"

if quality_summary_path.exists():
    quality_summary = pd.read_csv(quality_summary_path)
    print("Quality summary rows", len(quality_summary))
    display(quality_summary)

if quality_report_path.exists():
    quality_report = pd.read_csv(quality_report_path)
    print("Quality report rows", len(quality_report))
    display(quality_report.head(20))


## 7. Spectrogram Tensor Construction

Each valid event is converted from audio to a fixed-size tensor in `src.data.spectrogram`.

Processing steps:

1. Load the WAV file.
2. Resample to the configured sample rate if necessary.
3. Convert multi-channel audio to mono.
4. Crop the event with a configurable time margin.
5. Compute a mel spectrogram and convert amplitude to decibels.
6. Normalize either per sample or with configured global dB bounds.
7. Resize to `model.img_size`.
8. Build a frequency mask from annotation low/high frequency.
9. Stack three channels: full spectrogram, mask, highlighted spectrogram.

Because the model receives the annotation frequency band as input, inference is designed for already localized/annotated events, not arbitrary unsegmented raw audio.

In [ ]:

from src.data.spectrogram import event_tensor

sample_tensor_available = False
sample_row = manifest[manifest["split"] == config.get("val_split", "validation")].iloc[0].to_dict()
print({key: sample_row[key] for key in ["split", "dataset", "filename", "label", "low_frequency", "high_frequency"]})

try:
    tensor = event_tensor(sample_row, config["audio"], int(config["model"]["img_size"]))
except (ImportError, OSError, RuntimeError, SystemExit) as exc:
    tensor = None
    print(f"Skipped sample tensor generation: {type(exc).__name__}: {exc}")
else:
    sample_tensor_available = True
    print("tensor shape", tuple(tensor.shape))
    print("min", float(tensor.min()), "max", float(tensor.max()), "dtype", tensor.dtype)


In [ ]:

import matplotlib.pyplot as plt

if sample_tensor_available:
    titles = ["full spectrogram", "frequency mask", "highlighted band"]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, channel, title in zip(axes, tensor, titles):
        axis.imshow(channel.numpy(), cmap="magma", origin="upper")
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
else:
    print("Skipped spectrogram plot because no sample tensor is available.")


## 8. Tensor Cache

`cached_event_tensor` stores generated event tensors under `processed_cache/<split>/<label>/<hash>.pt`.

The cache key includes transform version, dtype, split, dataset, filename, audio path, audio file size, audio modification time, event times, frequency band, audio config, and image size. This invalidates stale cache entries when audio files or preprocessing settings change.

In [ ]:

if RUN_CACHE_SUMMARY:
    cache = pipeline.cache_summary(CACHE_ROOT)
else:
    print("Skipped cache summary.")


## 9. Dataset And DataLoader

`BioacousticDataset` filters the manifest by split, validates labels against the configured class list, loads image tensors or spectrogram tensors, and returns `(tensor, label_index, metadata)`.

`create_loader` wraps the dataset in a PyTorch DataLoader. For training, it can optionally use `WeightedRandomSampler` based on inverse class frequency and configured class multipliers.

In [ ]:

from src.data.dataset import BioacousticDataset

try:
    dataset = BioacousticDataset(
        manifest_path=MANIFEST_PATH,
        split=config.get("val_split", "validation"),
        class_names=config["classes"],
        img_size=int(config["model"]["img_size"]),
        audio_cfg=config["audio"],
        train=False,
        mode=config.get("dataset", {}).get("mode", "spectrogram"),
        cache_cfg=None,
    )
    print("dataset length", len(dataset))
    print("class counts", dict(zip(config["classes"], dataset.class_counts())))
    x, y, meta = dataset[0]
except (ImportError, OSError, RuntimeError, SystemExit) as exc:
    print(f"Skipped dataset sample load: {type(exc).__name__}: {exc}")
else:
    print("sample tensor", tuple(x.shape), "label index", y, "label", config["classes"][y])
    print("metadata keys", sorted(meta.keys()))


## 10. Model

The model factory currently supports `resnet18`. It uses torchvision's ResNet18 and replaces the final fully connected layer with `num_classes` outputs.

The first convolution is unchanged, which matches the 3-channel spectrogram tensor.

In [ ]:
try:
    from src.models.resnet import create_model
    model = create_model(
        name=config["model"]["name"],
        num_classes=len(config["classes"]),
        pretrained=bool(config["model"].get("pretrained", False)),
    )
except (ImportError, OSError, RuntimeError, SystemExit) as exc:
    model = None
    print(f"Skipped model inspection: {type(exc).__name__}: {exc}")
else:
    print(model.__class__.__name__)
    print("final layer", model.fc)


## 11. Training

`src.training.train` performs the training loop.

Main operations:

- set random seeds;
- select CUDA if requested and available, otherwise CPU;
- create a timestamped run directory;
- save the resolved config;
- create train and validation loaders;
- create the ResNet18 classifier;
- compute class weights from train-set class counts;
- optionally apply class-specific multipliers;
- choose cross-entropy or focal loss;
- optimize with AdamW;
- reduce learning rate on validation macro-F1 plateau;
- use automatic mixed precision on CUDA;
- save best and last checkpoints;
- run final evaluation for the best checkpoint.

The notebook does not run full training unless `RUN_FULL_TRAINING = True`.

In [ ]:

last_run_dir = None
if RUN_SMOKE_TRAINING:
    last_run_dir = pipeline.train(config_path=SMOKE_CONFIG_PATH, manifest=MANIFEST_PATH)
elif RUN_FULL_TRAINING:
    last_run_dir = pipeline.train(config_path=CONFIG_PATH, manifest=MANIFEST_PATH)
else:
    print("Skipped training. Existing runs can be inspected in outputs/runs/.")


## 12. Existing Runs

Each run directory may contain checkpoints, config copy, history, metrics, predictions, confusion matrices, and error reports.

In [ ]:
runs_root = PROJECT_ROOT / "outputs" / "runs"
run_rows = []
for metrics_path in sorted(runs_root.glob("*/best_metrics.json")):
    metrics = json.loads(metrics_path.read_text())
    run_rows.append({
        "run": metrics_path.parent.name,
        "accuracy": metrics.get("accuracy"),
        "balanced_accuracy": metrics.get("balanced_accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "weighted_f1": metrics.get("weighted_f1"),
    })

runs = pd.DataFrame(run_rows).sort_values("macro_f1", ascending=False)
display(runs)
best_run = runs.iloc[0]["run"] if len(runs) else None
print("best run by macro_f1", best_run)


## 13. Evaluation

`src.training.evaluate` loads a checkpoint, runs inference over the selected split, and writes:

- `best_metrics.json`;
- `classification_report.csv`;
- `metrics_by_dataset.csv`;
- `metrics_by_dataset_class.csv`;
- `baseline_metrics.csv`;
- `class_confidence_analysis.csv`;
- `pr_curves.csv`;
- `error_analysis.csv`;
- `top_confusion_pairs.csv`;
- focused error reports for `bpd/bmd` and `bmb/bmz`;
- confusion matrix CSV and PNG files;
- `val_predictions.csv` with one row per evaluated event.

The main global metric is macro-F1. Per-dataset reports include `macro_f1_present_classes`, which ignores classes absent from a dataset.

In [ ]:

if RUN_EVALUATION:
    if CHECKPOINT_PATH is not None:
        checkpoint = Path(CHECKPOINT_PATH)
        output_dir = Path(EVALUATION_OUTPUT_DIR or checkpoint.parent)
    elif best_run:
        checkpoint = RUNS_ROOT / best_run / "best_model.pt"
        output_dir = Path(EVALUATION_OUTPUT_DIR or RUNS_ROOT / best_run)
    else:
        raise RuntimeError("No checkpoint selected for evaluation.")
    evaluation_metrics = pipeline.evaluate(
        checkpoint=checkpoint,
        config_path=CONFIG_PATH,
        manifest=MANIFEST_PATH,
        output_dir=output_dir,
    )
else:
    print("Skipped evaluation. Existing evaluation files are inspected below.")


In [ ]:
if best_run:
    best_dir = runs_root / best_run
    metrics = json.loads((best_dir / "best_metrics.json").read_text())
    print("run", best_run)
    for key in ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]:
        if key in metrics:
            print(f"{key:22s} {metrics[key]:.4f}")

    report = pd.read_csv(best_dir / "classification_report.csv")
    display(report[report["label"].isin(config["classes"])])

    dataset_metrics_path = best_dir / "metrics_by_dataset.csv"
    if dataset_metrics_path.exists():
        display(pd.read_csv(dataset_metrics_path))


## 14. Error Analysis

`error_analysis.csv` counts true-label/predicted-label pairs for misclassified events. Focused reports isolate error families that were important in the experiments, especially `bpd` versus `bmd` and `bmb` versus `bmz`.

In [ ]:
if best_run:
    best_dir = runs_root / best_run
    for name in ["error_analysis.csv", "bpd_error_report.csv", "bmb_bmz_error_report.csv", "baseline_metrics.csv", "class_confidence_analysis.csv"]:
        path = best_dir / name
        if path.exists():
            print(name)
            display(pd.read_csv(path).head(15))


## 15. Inference

`src.training.predict` supports two inference modes:

- image mode: classify an already generated RGB spectrogram image;
- audio mode: classify one WAV event using start/end seconds and low/high frequency metadata.

Audio mode follows the same tensor construction as training.

In [ ]:

if best_run:
    example = manifest[manifest["split"] == config.get("val_split", "validation")].iloc[0]
    checkpoint = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH is not None else RUNS_ROOT / best_run / "best_model.pt"
    if RUN_PREDICTION:
        prediction = pipeline.predict(
            checkpoint=checkpoint,
            config_path=CONFIG_PATH,
            audio=example.audio_path,
            start_seconds=float(example.clip_start_seconds),
            end_seconds=float(example.clip_end_seconds),
            low_frequency=float(example.low_frequency),
            high_frequency=float(example.high_frequency),
        )
    else:
        print("Prepared prediction inputs. Set RUN_PREDICTION = True to execute.")
        print("checkpoint", checkpoint)
        print("audio", example.audio_path)


## 16. Analysis Utilities

The repository includes two post-training utilities:

- `src.analysis.imbalance_audit`: audits class imbalance, leakage, baselines, and report completeness.
- `src.analysis.inspect_errors`: exports PNG visualizations for selected error-report rows.

These scripts consume artifacts produced by evaluation.

In [ ]:

if RUN_IMBALANCE_AUDIT:
    if not best_run:
        raise RuntimeError("No evaluated run found for imbalance audit.")
    pipeline.imbalance_audit(
        config_path=CONFIG_PATH,
        manifest=MANIFEST_PATH,
        run_dir=RUNS_ROOT / best_run,
    )
else:
    print("Skipped imbalance audit. Set RUN_IMBALANCE_AUDIT = True to execute.")

if RUN_ERROR_EXPORT:
    if not best_run:
        raise RuntimeError("No evaluated run found for error export.")
    report = Path(ERROR_REPORT_PATH) if ERROR_REPORT_PATH is not None else RUNS_ROOT / best_run / "bpd_error_report.csv"
    pipeline.inspect_errors(
        report=report,
        config_path=CONFIG_PATH,
        out=ERROR_EXPORT_DIR,
        limit=40,
    )
else:
    print("Skipped error export. Set RUN_ERROR_EXPORT = True to execute.")


## 17. Tests

The test suite verifies objective behavior for config inheritance, audio datetime parsing, manifest filtering, metric handling for absent classes, cache invalidation, focal loss, weighted sampling, and evaluation report writers.

In [ ]:

if RUN_TESTS:
    pipeline.run_tests(("-q",))
else:
    print("Skipped tests.")


## 18. Current Objective Summary

Based on the existing project artifacts:

- The manifest contains all valid events used by training and validation.
- Training and validation are separated by dataset source, not by random stratification.
- The best documented run is selected by macro-F1.
- The strongest recurring error families are visible in `error_analysis.csv`.
- Cache size should be monitored because one cached tensor is stored per manifest event and transform setting.
- The frequency-band metadata is part of the model input, so deployment assumptions must include event localization and frequency bounds.